# SFVP AI Clone — Notebook 03: Photo Animation (LivePortrait)
**Tool:** LivePortrait (Apache 2.0 — commercial use OK)

Use this when you don't have a suitable base video — animates a still portrait photo of Shennel using the cloned voice audio. The photo comes alive with natural head movements and facial expressions.

**Drive structure needed:**
```
My Drive/SFVP_Clone/
├── photos/      ← portrait photos of Shennel (.jpg/.png)
└── outputs/     ← audio from Notebook 01 + video outputs here
```

**Runtime:** T4 GPU

In [ ]:
# CELL 1 — Install LivePortrait (run once per session, ~3 min)
import os

if not os.path.exists('/content/LivePortrait'):
    !git clone https://github.com/KwaiVGI/LivePortrait.git /content/LivePortrait -q

%cd /content/LivePortrait
!pip install -r requirements.txt -q
!apt-get install -y ffmpeg -q
print('LivePortrait installed.')

In [ ]:
# CELL 2 — Download LivePortrait model weights
from huggingface_hub import snapshot_download

print('Downloading LivePortrait weights (~1.5GB, cached after first run)...')
snapshot_download(
    repo_id='KwaiVGI/LivePortrait',
    local_dir='/content/LivePortrait/pretrained_weights',
    ignore_patterns=['*.md', '*.txt']
)
print('Weights downloaded.')

In [ ]:
# CELL 3 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/SFVP_Clone'

# List photos
photos_dir = f'{DRIVE_BASE}/photos'
if os.path.exists(photos_dir):
    photos = [f for f in os.listdir(photos_dir) if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))]
    print(f'Found {len(photos)} photo(s):')
    for i, p in enumerate(photos):
        print(f'  [{i}] {p}')

# List audio
audios = [f for f in os.listdir(f'{DRIVE_BASE}/outputs') if f.endswith('.wav')]
print(f'\nAvailable audio files:')
for a in audios:
    print(f'  {a}')

In [ ]:
# CELL 4 — CONFIGURE THIS CELL

# Portrait photo to animate
PHOTO_FILENAME = 'shennel_portrait_01.jpg'  # <-- CHANGE THIS

# Voice audio to animate to (from Notebook 01 outputs)
AUDIO_FILENAME = 'voice_output_01.wav'  # <-- CHANGE THIS

# Output video filename
OUTPUT_FILENAME = 'photo_anim_01.mp4'  # <-- CHANGE IF NEEDED

# Expression scale: 1.0 = natural, 1.3 = more expressive, 0.7 = subtle
EXPRESSION_SCALE = 1.0

# Enable stitching for smoother edges around the face
FLAG_STITCHING = True

print('Config set.')

In [ ]:
# CELL 5 — Generate a driving video from the audio
# LivePortrait needs a driving video (showing head movement pattern).
# We'll create a simple neutral driving video from a sample, then sync audio to it.
import subprocess
import shutil

# Copy photo and audio to local
local_photo = '/content/source_portrait.jpg'
local_audio = '/content/audio.wav'
shutil.copy(f'{DRIVE_BASE}/photos/{PHOTO_FILENAME}', local_photo)
shutil.copy(f'{DRIVE_BASE}/outputs/{AUDIO_FILENAME}', local_audio)

# Get audio duration
result = subprocess.run(
    ['ffprobe', '-v', 'quiet', '-show_entries', 'format=duration', '-of', 'csv=p=0', local_audio],
    capture_output=True, text=True
)
duration = float(result.stdout.strip())
print(f'Audio duration: {duration:.1f}s')

# Download a neutral talking driving video template (head movements only)
# Using a publicly available driving template from the LivePortrait examples
driving_video = '/content/driving.mp4'
if not os.path.exists(driving_video):
    !wget -q -O /content/driving.mp4 https://github.com/KwaiVGI/LivePortrait/raw/main/assets/examples/driving/d0.mp4

# Loop driving video to match audio duration
fps = 25
n_frames = int(duration * fps) + 10
driving_looped = '/content/driving_looped.mp4'
subprocess.run([
    'ffmpeg', '-y', '-stream_loop', '-1', '-i', driving_video,
    '-frames:v', str(n_frames), '-c:v', 'libx264', '-an', driving_looped
], capture_output=True)
print(f'Driving video prepared ({n_frames} frames @ {fps}fps).')

In [ ]:
# CELL 6 — Run LivePortrait animation
%cd /content/LivePortrait

anim_output = '/content/animated_no_audio.mp4'

cmd = [
    'python', 'inference.py',
    '-s', local_photo,
    '-d', driving_looped,
    '--output_dir', '/content/lp_output',
    '--expression_scale', str(EXPRESSION_SCALE),
]
if FLAG_STITCHING:
    cmd.append('--flag_stitching')

print('Running LivePortrait animation...')
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    print('ERROR:', result.stderr[-2000:])
else:
    # Find output file
    import glob
    outputs = glob.glob('/content/lp_output/*.mp4')
    if outputs:
        shutil.copy(outputs[0], anim_output)
        print(f'Animation done: {anim_output}')
    else:
        print('ERROR: No output file found. Check error logs.')

In [ ]:
# CELL 7 — Merge audio into animated video and save to Drive
local_final = '/content/photo_anim_final.mp4'
drive_final = f'{DRIVE_BASE}/outputs/{OUTPUT_FILENAME}'

# Merge audio with animated video, trim to audio length
subprocess.run([
    'ffmpeg', '-y',
    '-i', anim_output,
    '-i', local_audio,
    '-c:v', 'libx264', '-c:a', 'aac',
    '-shortest',
    local_final
], capture_output=True)

shutil.copy(local_final, drive_final)
print(f'Final video with audio saved to: {drive_final}')

In [ ]:
# CELL 8 — Preview
from IPython.display import HTML
from base64 import b64encode

with open(local_final, 'rb') as f:
    vb64 = b64encode(f.read()).decode()

HTML(f'''
<video width="480" controls>
  <source src="data:video/mp4;base64,{vb64}" type="video/mp4">
</video>
<p><b>Tips:</b><br>
- Increase EXPRESSION_SCALE (try 1.2) for more lively expressions<br>
- Use a different portrait photo for different looks<br>
- Send this to Notebook 04 to enhance and prepare for posting</p>
''')